# NUS Prerequisite Graph Processing

This notebook processes NUS course prerequisites and saves the resulting graph as JSON.

In [1]:
import os
from pathlib import Path

# Get notebook directory and construct data path
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
BASE_DATA_PATH = os.path.join(NOTEBOOK_DIR, "..", "DSA4264 Data")
COURSES_DIR = os.path.join(BASE_DATA_PATH, "NUS-SMU-SUTD Courses")
NUS_COURSES_CSV = os.path.join(COURSES_DIR, "nus_courses.csv")
DEPENDENT_MODS_FILE = os.path.join(BASE_DATA_PATH, "dependent_mods_dedup.csv")
OUTPUT_JSON = os.path.join(BASE_DATA_PATH, "nus_prerequisite_graph.json")

In [2]:
# Imports
import pandas as pd
import numpy as np
import json
import ast

In [3]:
# Load NUS courses
nus_courses_df = pd.read_csv(NUS_COURSES_CSV)
nus_courses_df.columns = nus_courses_df.columns.str.strip().str.lower()
nus_modules = set(nus_courses_df['code'].dropna().astype(str))
print(f"Loaded {len(nus_modules)} unique NUS modules")

# Load prerequisite dependencies
prereq_df = pd.read_csv(DEPENDENT_MODS_FILE)
prereq_df.columns = prereq_df.columns.str.strip().str.lower()
print(f"Loaded {len(prereq_df)} prerequisite relationships")

# Remove % signs from prereq_token
if 'prereq_token' in prereq_df.columns:
    prereq_df['prereq_token'] = prereq_df['prereq_token'].str.replace('%', '', regex=False)

# Parse dependent_modules_dedup from string lists
def parse_module_list(val):
    if pd.isna(val) or val == '':
        return []
    try:
        return ast.literal_eval(val)
    except:
        return []

prereq_df['dependent_modules_dedup'] = prereq_df['dependent_modules_dedup'].apply(parse_module_list)

# Filter to only NUS modules
prereq_df_nus = prereq_df[
    prereq_df['prereq_token'].isin(nus_modules) &
    prereq_df['dependent_modules_dedup'].apply(lambda deps: any(d in nus_modules for d in deps))
].copy()

print(f"Filtered to {len(prereq_df_nus)} NUS prerequisite relationships")

# Build unlocks graph: prereq_token -> list of modules it unlocks
unlocks_graph = {}
for _, row in prereq_df_nus.iterrows():
    prereq = row['prereq_token']
    dependents = [d for d in row['dependent_modules_dedup'] if d in nus_modules]
    if prereq not in unlocks_graph:
        unlocks_graph[prereq] = []
    unlocks_graph[prereq].extend(dependents)

# Remove duplicates
for k in unlocks_graph:
    unlocks_graph[k] = list(set(unlocks_graph[k]))

print(f"\nBuilt prerequisite graph with {len(unlocks_graph)} nodes")

# Display top 5 foundational modules
sorted_prereqs = sorted(unlocks_graph.items(), key=lambda x: len(x[1]), reverse=True)
print("\nTop 5 foundational modules (unlock most courses):")
for i, (code, unlocks) in enumerate(sorted_prereqs[:5], 1):
    print(f"{i}. {code:10s} unlocks {len(unlocks):3d} modules")

# Save to JSON
with open(OUTPUT_JSON, 'w') as f:
    json.dump(unlocks_graph, f, indent=2)

print(f"\n✓ Saved prerequisite graph to: {os.path.basename(OUTPUT_JSON)}")

Loaded 381 unique NUS modules
Loaded 1977 prerequisite relationships
Filtered to 83 NUS prerequisite relationships

Built prerequisite graph with 81 nodes

Top 5 foundational modules (unlock most courses):
1. ST2334     unlocks  18 modules
2. ST2131     unlocks  17 modules
3. MA2116     unlocks  17 modules
4. MA2002     unlocks  14 modules
5. MA2001     unlocks  12 modules

✓ Saved prerequisite graph to: nus_prerequisite_graph.json
